In [1]:
%cd ../../../

/Users/hoangle/Projects/fwo_models


In [2]:
from pathlib import Path
import datetime

import joblib
import numpy as np
import polars as pl
import lightning as L
import torch.nn.functional as F
from torch import nn
from torch.optim import AdamW
from sklearn.metrics import root_mean_squared_error, r2_score
from torch.utils.data import Dataset, DataLoader
import torch
from torch.nn import Module
from torch import Tensor
from torchmetrics.regression import MeanSquaredError, R2Score
from lightning.pytorch.callbacks import RichProgressBar
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.preprocessing import MinMaxScaler
from polars import DataFrame

# Load data and resources

In [3]:
path_dir = Path("data/inter/idea7")

In [4]:
path_train = path_dir / "train.parquet"
path_val = path_dir / "val.parquet"

pos_train = pl.read_parquet(path_train)
pos_val = pl.read_parquet(path_val)

In [5]:
path_scaler = path_dir / "pcs_scaler.gz"

scaler_pcs = joblib.load(path_scaler)

In [6]:
path_embds = path_dir / "meal_embds.npy"
embds = np.load(path_embds)

# Define models, dataloader

## Define Data module

In [7]:
def _fill_none(x):
    return x if x is not None else 0

def _create_tensor(col: str, record: dict):
    return torch.tensor(
        [
            _fill_none(record[f'{col}_1']),
            _fill_none(record[f'{col}_2']),
            _fill_none(record[f'{col}_3']),
            _fill_none(record[f'{col}_4']),
        ],
        dtype=torch.float32
    )

def _create_mask(record: dict):
    return torch.tensor(
        [
            record['dist_1'] is None,
            record['dist_2'] is None,
            record['dist_3'] is None,
            record['dist_4'] is None,
        ],
        dtype=torch.bool
    )

class POSData(Dataset):
    def __init__(self, ds: DataFrame) -> None:
        super().__init__()

        self._ds = ds

    def __getitem__(self, idx):
        record = self._ds.row(idx, named=True)

        restaurant = torch.tensor(record['restaurant_enc'], dtype=torch.int32)

        meal = _create_tensor('meal_id_sim_enc', record).type(torch.int32)
        meal_type = _create_tensor('meal_type_enc', record).type(torch.int32)
        serv_pcn = _create_tensor('serving_percent', record)
        sim = _create_tensor('dist', record)
        tgt = _create_tensor('pcs_scaled', record)

        mask = _create_mask(record)

        date = torch.tensor(
            [record['weekday_sin'], record['weekday_cos'], record['day_sin'],
            record['day_cos'], record['month_sin'],record['month_cos'],],
            dtype=torch.float32
        )


        out = {
            'meal': meal,
            'meal_type': meal_type,
            'sim': sim,
            'mask': mask,
            'restaurant': restaurant,
            'date': date,
            'serv_pcn': serv_pcn,
            'tgt': tgt
        }

        return out

    def __len__(self,) -> int:
        return len(self._ds)

# train_ds = POSData(pos_train.filter(pl.col('meal_id_sim_enc_4').is_null()))
# # X = train_ds[0]
# loader_train = DataLoader(train_ds, batch_size=10, shuffle=True)
# for X in loader_train:
#     break

# X['mask']

## Define model

In [8]:
class POSForecast(Module):
    def __init__(
        self,
        n_meal_types: int = 5,
        n_restaurants: int = 4,
        n_meals: int = 149,
        d_hid: int = 32,
        d_raw_meal_emd: int = 1024,
    ) -> None:
        super().__init__()

        self._embd_meal_type = nn.Embedding(n_meal_types, d_hid)
        self._embd_restaurant = nn.Embedding(n_restaurants, d_hid)
        self._embd_meal = nn.Embedding(n_meals, d_raw_meal_emd)

        self.lin_date = nn.Linear(6, d_hid)
        self.lin_sim = nn.Linear(1, d_hid)
        self.lin_serv = nn.Linear(1, d_hid)
        self.lin_meal = nn.Linear(d_raw_meal_emd, d_hid)

        self.ff_combine = nn.Sequential(
            nn.Linear(d_hid * 6, d_hid * 6),
            nn.Dropout(),
            nn.Tanh(),
            nn.LayerNorm(d_hid * 6),
        )

        self.trans_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_hid * 6, nhead=1, batch_first=True),
            num_layers=2,
            enable_nested_tensor=False,
        )

        self.lin_pcs = nn.Linear(d_hid * 6, 1)

    def forward(self, X: dict[str, Tensor]) -> Tensor:
        meal = X["meal"]
        meal_type = X["meal_type"]
        mask = X["mask"]
        restaurant = X["restaurant"]
        date = X["date"]
        sim = X["sim"]
        serv_pcn = X['serv_pcn']

        # Encode
        meal_type = self._embd_meal_type(meal_type)
        restaurant = self._embd_restaurant(restaurant)
        meal = self._embd_meal(meal)
        meal = self.lin_meal(meal)

        date = self.lin_date(date)
        sim = self.lin_sim(sim.unsqueeze(-1))
        serv_pcn = self.lin_serv(serv_pcn.unsqueeze(-1))

        # Concate fields
        N = meal.shape[1]
        restaurant = torch.repeat_interleave(restaurant.unsqueeze(1), N, dim=1)
        date = torch.repeat_interleave(date.unsqueeze(1), N, dim=1)

        meals = torch.concat(
            [
                meal,
                meal_type,
                restaurant,
                date,
                sim,
                serv_pcn,
            ],
            dim=-1,
        )
        meals = self.ff_combine(meals)

        # Use Transformer Encoder
        # SEQ_LEN = mask.shape[-1]
        # mask = mask.unsqueeze(1).repeat_interleave(SEQ_LEN, dim=1)
        meals = self.trans_encoder(meals, src_key_padding_mask=mask)

        # meal_main = meals[:, 0:1]       # [bz, 1, d_hid * 4]
        # meals_other = meals[:, 1:]      # [bz, N-1, d_hid * 4]
        # S: Tensor = meal_main @ meals_other.permute(0, 2, 1)        # [bz, 1, N-1]
        # attentive_prob = nn.functional.softmax(S.masked_fill_(mask.unsqueeze(1), -1e10), dim=-1)
        # # [bz, 1, N-1]
        # h = attentive_prob @ meals_other
        # # [bz, 1, d_hid * 4]

        # meal_main = torch.concat([meal_main, h], dim=-1)
        # meal_main = self.lin2(meal_main)
        # [bz, 1, d_hid * 8]

        # Predict pos
        pos = self.lin_pcs(meals).squeeze(-1)
        # pos = F.tanh(pos)
        
        return pos





# train_ds = POSData(pos_train.filter(pl.col('serving_percent_4').is_null()))
# # X = train_ds[0]
# loader_train = DataLoader(train_ds, batch_size=10, shuffle=True)
# for X in loader_train:
#     break

# model = POSForecast()

# # out = model(X)
# # out


# meal = X["meal"]
# meal_type = X["meal_type"]
# mask = X["mask"]
# restaurant = X["restaurant"]
# date = X["date"]
# sim = X["sim"]
# serv_pcn = X['serv_pcn']

# # Encode
# meal_type = model._embd_meal_type(meal_type)
# restaurant = model._embd_restaurant(restaurant)
# meal = model._embd_meal(meal)
# meal = model.lin_meal(meal)

# date = model.lin_date(date)
# sim = model.lin_sim(sim.unsqueeze(-1))
# serv_pcn = model.lin_serv(serv_pcn.unsqueeze(-1))

# # Concate fields
# N = meal.shape[1]
# restaurant = torch.repeat_interleave(restaurant.unsqueeze(1), N, dim=1)
# date = torch.repeat_interleave(date.unsqueeze(1), N, dim=1)

# meals = torch.concat(
#     [
#         meal,
#         meal_type,
#         restaurant,
#         date,
#         sim,
#         serv_pcn,
#     ],
#     dim=-1,
# )
# meals = model.ff_combine(meals)
# # SEQ_LEN = mask.shape[-1]
# # mask = mask.unsqueeze(1).repeat_interleave(SEQ_LEN, dim=1)
# meals = model.trans_encoder(meals, src_key_padding_mask=mask)

In [38]:
class LitPOSForecast(L.LightningModule):
    def __init__(
        self,
        scaler,
        params: dict,
        lr: float = 3e-4,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        self.scaler = scaler

        self.forecaster = POSForecast(**params)
        self.lr = lr

        self.mse = MeanSquaredError()
        self.r2 = R2Score()
        self.preds_val, self.tgts_val = [], []
        self.preds_train, self.tgts_train = [], []

    def training_step(self, batch, batch_idx):
        tgt = batch["tgt"]

        pred = self.forecaster(batch)

        # pred = ((~batch['mask']).type(torch.float32) + EPS) * pred
        # tgt = ((~batch['mask']).type(torch.float32) + EPS) * tgt

        loss = nn.functional.mse_loss(pred, tgt)
        self.log("train_loss", loss, prog_bar=True, on_step=True)

        self.preds_train.append(pred)
        self.tgts_train.append(tgt)

        return loss

    def on_train_epoch_end(self) -> None:
        preds = self.scaler.inverse_transform(torch.concat(self.preds_train, dim=0).detach().cpu()).flatten()
        tgts = self.scaler.inverse_transform(torch.concat(self.tgts_train, dim=0).detach().cpu()).flatten()

        rmse = root_mean_squared_error(preds, tgts)
        r2 = r2_score(preds, tgts)

        self.log("rmse_train", rmse, on_epoch=True)
        self.log("r2_train", r2, on_epoch=True)

        self.preds_train, self.tgts_train = [], []

    def validation_step(self, batch, batch_idx):
        tgt = batch["tgt"]

        pred = self.forecaster(batch)

        self.preds_val.append(pred)
        self.tgts_val.append(tgt)

    def on_validation_epoch_end(self) -> None:
        preds = self.scaler.inverse_transform(torch.concat(self.preds_val, dim=0).detach().cpu()).flatten()
        tgts = self.scaler.inverse_transform(torch.concat(self.tgts_val, dim=0).detach().cpu()).flatten()

        rmse = root_mean_squared_error(preds, tgts)
        r2 = r2_score(preds, tgts)
        # rmse = torch.sqrt(self.mse(preds, tgts))
        # r2 = self.r2(preds, tgts)

        self.log("rmse_val", rmse, on_epoch=True)
        self.log("r2_val", r2, on_epoch=True)

        self.preds_val, self.tgts_val = [], []

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.lr)

        return optimizer


# Train

In [41]:
BATCH_SIZE = 128
LR = 3e-4

params = {
    'n_meal_types': 5,
    'n_restaurants': 4,
    'n_meals': 149,
    'd_hid': 32,
    'd_raw_meal_emd': 1024,
}

loader_train = DataLoader(POSData(pos_train), batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(POSData(pos_val), batch_size=BATCH_SIZE)

# litmodel = LitPOSForecast(scaler_pcs, params, LR)
# litmodel.forecaster._embd_meal.weight = nn.parameter.Parameter(torch.tensor(embds, dtype=torch.float32), requires_grad=True)
litmodel = LitPOSForecast.load_from_checkpoint("weights/idea7.ckpt")

version = datetime.datetime.now().strftime("%m-%d_%H-%M-%S")
trainer = L.Trainer(
    # devices=0,
    callbacks=[RichProgressBar(leave=True)],
    logger=TensorBoardLogger("tb_logs", name="idea7_dl_topK", version=version),
    gradient_clip_val=1,
    max_epochs=20,
)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [40]:
# trainer.save_checkpoint("weights/idea7.ckpt")
litmodel = LitPOSForecast.load_from_checkpoint("weights/idea7.ckpt")

In [42]:
# trainer.fit(litmodel, loader_train, loader_val)
trainer.validate(litmodel, loader_val)

Output()

/Users/hoangle/Projects/fwo_models/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          r2_val           │    0.5520349740982056     │
│         rmse_val          │     64.38379669189453     │
└───────────────────────────┴───────────────────────────┘

[{'rmse_val': 64.38379669189453, 'r2_val': 0.5520349740982056}]

In [35]:
preds = litmodel.scaler.inverse_transform(torch.concat(litmodel.preds_val, dim=0).detach().cpu()).flatten()
tgts = litmodel.scaler.inverse_transform(torch.concat(litmodel.tgts_val, dim=0).detach().cpu()).flatten()

rmse = root_mean_squared_error(preds, tgts)
r2 = r2_score(preds, tgts)
# rmse = torch.sqrt(litmodel.mse(preds, tgts))
# r2 = litmodel.r2(preds, tgts)

In [37]:
r2

0.6790726273770666

In [21]:
r2

0.6790726273770666

In [ ]:
preds = torch.tensor(scaler_pcs.inverse_transform(litmodel.forecaster(x).detach().numpy()).flatten())
tgt = torch.tensor(scaler_pcs.inverse_transform(x['tgt'].detach().numpy()).flatten())

litmodel.mse(preds, tgt).sqrt()

tensor(61.7532)